# Credit Risk Analyzer - Improved Version

## Overview
This notebook provides a comprehensive analysis of credit risk using application and previous application data.
It includes data exploration, cleaning, feature engineering, and analysis.

## Table of Contents
1. [Setup & Configuration](#setup)
2. [Data Loading](#loading)
3. [Exploratory Data Analysis](#eda)
4. [Data Cleaning](#cleaning)
5. [Feature Engineering](#features)
6. [Analysis & Insights](#analysis)

## 1. Setup & Configuration

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Suppress warnings
warnings.filterwarnings('ignore')

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Libraries imported successfully!')

## 2. Data Loading

In [ ]:
# Define data paths
data_path = Path('.')
application_file = data_path / 'application_data.csv'
previous_app_file = data_path / 'previous_application.csv'

# Load datasets
try:
    df_application = pd.read_csv(application_file)
    df_previous = pd.read_csv(previous_app_file)
    print(f'✓ Application data loaded: {df_application.shape}')
    print(f'✓ Previous application data loaded: {df_previous.shape}')
except FileNotFoundError as e:
    print(f'✗ Error loading files: {e}')

## 3. Exploratory Data Analysis

### 3.1 Dataset Overview

In [ ]:
# Application data shape and basic info
print('='*60)
print('APPLICATION DATA OVERVIEW')
print('='*60)
print(f'Shape: {df_application.shape}')
print(f'Memory Usage: {df_application.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nColumns: {len(df_application.columns)}')
print(f'Missing Values: {df_application.isnull().sum().sum()}')

In [ ]:
# Data types distribution
print('\nData Types Distribution:')
print(df_application.dtypes.value_counts())
print('\nFirst few rows:')
df_application.head()

In [ ]:
# Detailed column information
print('Column Information:')
col_info = pd.DataFrame({
    'Column': df_application.columns,
    'Type': df_application.dtypes,
    'Missing': df_application.isnull().sum(),
    'Missing %': (df_application.isnull().sum() / len(df_application) * 100).round(2),
    'Unique': df_application.nunique()
})
print(col_info.to_string())

### 3.2 Descriptive Statistics

In [ ]:
# Summary statistics for numerical columns
print('\nNumerical Columns Summary Statistics:')
df_application.describe().T

In [ ]:
# Target variable distribution
print('\nTarget Variable Distribution (Default Status):')
target_dist = df_application['TARGET'].value_counts()
target_pct = df_application['TARGET'].value_counts(normalize=True) * 100

for i in [0, 1]:
    print(f'  {i}: {target_dist[i]:,} ({target_pct[i]:.2f}%)')

# Visualize target distribution
fig, ax = plt.subplots(figsize=(10, 6))
df_application['TARGET'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Target (0=No Default, 1=Default)')
ax.set_ylabel('Count')
ax.set_xticklabels(['No Default', 'Default'], rotation=0)
plt.tight_layout()
plt.show()

### 3.3 Missing Values Analysis

In [ ]:
# Identify columns with missing values
missing_data = pd.DataFrame({
    'Column': df_application.columns,
    'Missing_Count': df_application.isnull().sum(),
    'Missing_Percent': (df_application.isnull().sum() / len(df_application) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)

print(f'Columns with missing values: {len(missing_data)}')
print('\nTop 20 columns with missing values:')
print(missing_data.head(20).to_string())

# Visualize top missing values
if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing_data.head(20).plot(x='Column', y='Missing_Percent', kind='barh', ax=ax, legend=False)
    ax.set_title('Top 20 Columns with Missing Values', fontsize=14, fontweight='bold')
    ax.set_xlabel('Missing Percentage (%)')
    plt.tight_layout()
    plt.show()

## 4. Data Cleaning

### 4.1 Age Conversion (DAYS_BIRTH to Years)

In [ ]:
# Create a working copy
df = df_application.copy()

# Convert DAYS_BIRTH to age in years
# Note: DAYS_BIRTH is negative, so we take absolute value
df['AGE_YEARS'] = (abs(df['DAYS_BIRTH']) / 365.25).round(2)

print('Age Conversion:')
print(f'  Original DAYS_BIRTH range: {df["DAYS_BIRTH"].min()} to {df["DAYS_BIRTH"].max()}')
print(f'  AGE_YEARS range: {df["AGE_YEARS"].min():.2f} to {df["AGE_YEARS"].max():.2f}')
print(f'  Mean age: {df["AGE_YEARS"].mean():.2f} years')
print(f'  Median age: {df["AGE_YEARS"].median():.2f} years')

# Visualize age distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['AGE_YEARS'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Age Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age (Years)')
axes[0].set_ylabel('Frequency')

df.boxplot(column='AGE_YEARS', ax=axes[1])
axes[1].set_title('Age Distribution (Boxplot)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Age (Years)')

plt.tight_layout()
plt.show()

### 4.2 Employment Duration Handling

In [ ]:
# Analyze DAYS_EMPLOYED
print('DAYS_EMPLOYED Analysis:')
print(f'  Min: {df["DAYS_EMPLOYED"].min()}')
print(f'  Max: {df["DAYS_EMPLOYED"].max()}')
print(f'  Mean: {df["DAYS_EMPLOYED"].mean():.2f}')
print(f'  Median: {df["DAYS_EMPLOYED"].median():.2f}')

# Check for anomalies (unusual values like 365243)
anomaly_threshold = 365243
anomalies = df[df['DAYS_EMPLOYED'] > anomaly_threshold]
print(f'\n  Records with DAYS_EMPLOYED > {anomaly_threshold}: {len(anomalies)}')

# Create employment years column
df['EMPLOYMENT_YEARS'] = (abs(df['DAYS_EMPLOYED']) / 365.25).round(2)
df.loc[df['EMPLOYMENT_YEARS'] > 70, 'EMPLOYMENT_YEARS'] = np.nan  # Flag anomalies

print(f'\nEmployment Years (after cleaning):')
print(df['EMPLOYMENT_YEARS'].describe())

### 4.3 Categorical Variables Overview

In [ ]:
# Analyze categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns

print(f'Categorical Columns: {len(categorical_cols)}')
print('\nCategorical Variables Summary:')

for col in categorical_cols[:10]:  # Show first 10
    print(f'\n{col}:')
    print(f'  Unique values: {df[col].nunique()}')
    print(f'  Top 3 values:')
    print(df[col].value_counts().head(3).to_string())

## 5. Feature Engineering

### 5.1 Financial Ratios

In [ ]:
# Create financial ratios
print('Creating Financial Ratios...')

# Debt-to-Income Ratio
df['DEBT_TO_INCOME'] = (df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']).round(4)

# Credit-to-Income Ratio
df['CREDIT_TO_INCOME'] = (df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']).round(4)

# Annuity-to-Credit Ratio
df['ANNUITY_TO_CREDIT'] = (df['AMT_ANNUITY'] / df['AMT_CREDIT']).round(4)

# Goods Price Coverage (how well the credit covers the goods price)
df['PRICE_COVERAGE'] = (df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']).round(4)

print('✓ Financial ratios created successfully')
print('\nNew Financial Ratio Statistics:')

new_ratios = ['DEBT_TO_INCOME', 'CREDIT_TO_INCOME', 'ANNUITY_TO_CREDIT', 'PRICE_COVERAGE']
print(df[new_ratios].describe().T)

### 5.2 Age Groups Binning

In [ ]:
# Create age groups
age_bins = [0, 25, 35, 45, 55, 65, 150]
age_labels = ['<25', '25-35', '35-45', '45-55', '55-65', '>65']
df['AGE_GROUP'] = pd.cut(df['AGE_YEARS'], bins=age_bins, labels=age_labels, right=False)

print('Age Group Distribution:')
print(df['AGE_GROUP'].value_counts().sort_index())

# Visualize age groups
fig, ax = plt.subplots(figsize=(10, 6))
df['AGE_GROUP'].value_counts().sort_index().plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Distribution of Age Groups', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

## 6. Analysis & Insights

### 6.1 Target vs Age Analysis

In [ ]:
# Default rate by age group
print('Default Rate by Age Group:')
age_default = df.groupby('AGE_GROUP')['TARGET'].agg(['sum', 'count', 'mean'])
age_default.columns = ['Defaults', 'Total', 'Default_Rate']
age_default['Default_Rate'] = (age_default['Default_Rate'] * 100).round(2)
print(age_default)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_default['Default_Rate'].plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Default Rate by Age Group', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

df.boxplot(column='AGE_YEARS', by='TARGET', ax=axes[1]
        )
        axes[1].set_title('Age Distribution by Default Status', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Target (0=No Default, 1=Default)')
axes[1].set_ylabel('Age (Years)')
plt.suptitle('')
plt.tight_layout()
plt.show()

### 6.2 Income & Credit Analysis

In [ ]:
# Analyze income by default status
print('Income Statistics by Default Status:')
income_analysis = df.groupby('TARGET')[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']].describe().round(2)
print(income_analysis)

# Visualize income and credit
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, col in enumerate(['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']):
    df.boxplot(column=col, by='TARGET', ax=axes[idx])
    axes[idx].set_title(f'{col} by Default Status', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Target (0=No Default, 1=Default)')
    axes[idx].set_ylabel(col)

plt.suptitle('')
plt.tight_layout()
plt.show()

### 6.3 Contract Type Analysis

In [ ]:
# Contract type distribution
print('Contract Type Distribution:')
contract_dist = df['NAME_CONTRACT_TYPE'].value_counts()
print(contract_dist)

# Default rate by contract type
print('\nDefault Rate by Contract Type:')
contract_default = df.groupby('NAME_CONTRACT_TYPE')['TARGET'].agg(['sum', 'count', 'mean'])
contract_default.columns = ['Defaults', 'Total', 'Default_Rate']
contract_default['Default_Rate'] = (contract_default['Default_Rate'] * 100).round(2)
print(contract_default)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contract_dist.plot(kind='bar', ax=axes[0], color='lightblue', edgecolor='black')
axes[0].set_title('Distribution of Contract Types', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

contract_default['Default_Rate'].plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Default Rate by Contract Type', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### 6.4 Financial Ratios Impact on Default

In [ ]:
# Financial ratios by default status
print('Financial Ratios Statistics by Default Status:')
ratios_analysis = df.groupby('TARGET')[new_ratios].describe().round(4)
print(ratios_analysis)

# Visualize ratios
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(new_ratios):
    df.boxplot(column=col, by='TARGET', ax=axes[idx])
    axes[idx].set_title(f'{col} by Default Status', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Target (0=No Default, 1=Default)')
    axes[idx].set_ylabel(col)

plt.suptitle('')
plt.tight_layout()
plt.show()

## Summary of Key Findings

In [ ]:
print('='*70)
print('KEY FINDINGS FROM CREDIT RISK ANALYSIS')
print('='*70)

print(f'\n1. DATASET OVERVIEW')
print(f'   - Total Records: {len(df):,}')
print(f'   - Total Columns: {len(df.columns)}')
print(f'   - Default Rate: {(df["TARGET"].mean() * 100):.2f}%')

print(f'\n2. CUSTOMER DEMOGRAPHICS')
print(f'   - Average Age: {df["AGE_YEARS"].mean():.1f} years')
print(f'   - Age Range: {df["AGE_YEARS"].min():.1f} - {df["AGE_YEARS"].max():.1f} years')
print(f'   - Average Employment: {df["EMPLOYMENT_YEARS"].mean():.1f} years')

print(f'\n3. FINANCIAL PROFILE')
print(f'   - Average Income: ${df["AMT_INCOME_TOTAL"].mean():,.0f}')
print(f'   - Average Credit: ${df["AMT_CREDIT"].mean():,.0f}')
print(f'   - Average Annuity: ${df["AMT_ANNUITY"].mean():,.0f}')

print(f'\n4. RISK INDICATORS')
print(f'   - Avg Debt-to-Income Ratio: {df["DEBT_TO_INCOME"].mean():.4f}')
print(f'   - Avg Credit-to-Income Ratio: {df["CREDIT_TO_INCOME"].mean():.4f}')
print(f'   - Default Rate by Contract Type:')
for contract, rate in contract_default['Default_Rate'].items():
    print(f'     • {contract}: {rate:.2f}%')

print(f'\n5. AGE-BASED DEFAULT RISK')
for age_group, rate in age_default['Default_Rate'].items():
    print(f'   - {age_group}: {rate:.2f}%')

print('\n' + '='*70)

## Export Cleaned Data

In [ ]:
# Export cleaned dataset
output_path = Path('./cleaned_application_data.csv')
df.to_csv(output_path, index=False)
print(f'✓ Cleaned data exported to: {output_path}')
print(f'  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB')